### Import Library & Load Dataset
Import library yang diperlukan dan load dataset `IRIS_preprocessed.csv` yang sudah di-preprocessing pada post test 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

# load data yang udah dipreprocessing dari posttest 2
df = pd.read_csv('IRIS_preprocessed.csv')
df.head()

### Cek Data
Cek dulu bentuk datanya, berapa jumlah baris dan kolom, serta distribusi tiap spesies

In [ ]:
print(f"Jumlah baris: {df.shape[0]}, Jumlah kolom: {df.shape[1]}")
print()
print(df.dtypes)
print()
print(df['species'].value_counts()) # cek distribusi kelas

### Pisahkan Fitur (X) dan Target (y)
Menggunakan kolom numerik hasil preprocessing sebagai fitur (X) yaitu sepal_length, sepal_width, petal_length, petal_width, petal_area, dan petal_size_encoded. Kolom species jadi target (y).

In [ ]:
fitur = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'petal_area', 'petal_size_encoded']

X = df[fitur]
y = df['species']

print(f"Fitur: {list(X.columns)}")
print(f"Target: {y.unique()}")

### Split Data Training dan Testing
Data dibagi 80% training dan 20% testing, pake stratify supaya proporsi spesies tetap seimbang di kedua set.
Setelah itu dilakukan scaling menggunakan StandardScaler karena KNN dan SVM sensitif terhadap perbedaan skala antar fitur.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# standarisasi fitur
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # transform aja, jangan fit lagi

print(f"Data training: {X_train_scaled.shape[0]}")
print(f"Data testing: {X_test_scaled.shape[0]}")

### Training Model
Disini menggunakan 2 model klasifikasi:
- **KNN** (k=5) : klasifikasi berdasarkan 5 tetangga terdekat
- **SVM** (kernel rbf) : mencari hyperplane pemisah antar kelas

In [ ]:
# KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

# SVM
svm = SVC(kernel='rbf', random_state=42)
svm.fit(X_train_scaled, y_train)

print("model sudah dilatih")

### Evaluasi Model
Lakukan prediksi pada data testing kemudian evaluasi menggunakan accuracy dan classification report

In [ ]:
# prediksi
pred_knn = knn.predict(X_test_scaled)
pred_svm = svm.predict(X_test_scaled)

# --- KNN ---
print("=" * 35)
print("Evaluasi KNN")
print("=" * 35)
acc_knn = accuracy_score(y_test, pred_knn)
print(f"Accuracy: {acc_knn:.4f}")
print()
print(classification_report(y_test, pred_knn))

print()

# --- SVM ---
print("=" * 35)
print("Evaluasi SVM")
print("=" * 35)
acc_svm = accuracy_score(y_test, pred_svm)
print(f"Accuracy: {acc_svm:.4f}")
print()
print(classification_report(y_test, pred_svm))

### Perbandingan Akurasi
Bar chart untuk membandingkan akurasi KNN dan SVM

In [ ]:
nama_model = ['KNN', 'SVM']
akurasi = [acc_knn, acc_svm]

plt.figure(figsize=(6, 4))
bars = plt.bar(nama_model, akurasi, color=['#3498db', '#2ecc71'], width=0.5)

# tampilin nilai diatas bar
for bar, acc in zip(bars, akurasi):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{acc:.2%}', ha='center', fontweight='bold')

plt.ylim(0, 1.1)
plt.ylabel('Accuracy')
plt.title('Perbandingan Akurasi Model')
plt.show()

### Confusion Matrix
Menampilkan confusion matrix untuk masing-masing model supaya bisa dilihat detail prediksi yang benar dan salahnya

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# KNN
cm_knn = confusion_matrix(y_test, pred_knn)
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=knn.classes_, yticklabels=knn.classes_)
axes[0].set_title('Confusion Matrix KNN')
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Aktual')

# SVM
cm_svm = confusion_matrix(y_test, pred_svm)
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=svm.classes_, yticklabels=svm.classes_)
axes[1].set_title('Confusion Matrix SVM')
axes[1].set_xlabel('Prediksi')
axes[1].set_ylabel('Aktual')

plt.tight_layout()
plt.show()

### Scatter Plot Hasil Prediksi
Memvisualisasikan hasil prediksi model KNN pada data testing menggunakan 2 fitur yang paling berkorelasi tinggi yaitu petal_length dan petal_width. Warna menunjukkan hasil prediksi, sedangkan bentuk marker menunjukkan label aslinya.

In [ ]:
# dataframe buat plotting
df_plot = pd.DataFrame(X_test_scaled, columns=fitur)
df_plot['Aktual'] = y_test.values
df_plot['Prediksi'] = pred_knn
df_plot['Status'] = np.where(df_plot['Aktual'] == df_plot['Prediksi'], 'Benar', 'Salah')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# plot prediksi
sns.scatterplot(data=df_plot, x='petal_length', y='petal_width',
                hue='Prediksi', style='Aktual', s=100, ax=axes[0],
                palette='Set1')
axes[0].set_title('Hasil Prediksi KNN\n(warna=prediksi, bentuk=aktual)')
axes[0].set_xlabel('Petal Length (scaled)')
axes[0].set_ylabel('Petal Width (scaled)')
axes[0].legend(fontsize=8, loc='upper left')

# plot benar vs salah
warna = {'Benar': '#2ecc71', 'Salah': '#e74c3c'}
sns.scatterplot(data=df_plot, x='petal_length', y='petal_width',
                hue='Status', s=100, ax=axes[1], palette=warna)
axes[1].set_title('Prediksi Benar vs Salah (KNN)')
axes[1].set_xlabel('Petal Length (scaled)')
axes[1].set_ylabel('Petal Width (scaled)')

plt.tight_layout()
plt.show()

### Penjelasan Hasil Evaluasi

Dari hasil evaluasi kedua model menunjukkan performa yang baik. KNN dan SVM sama-sama mendapatkan akurasi yang tinggi pada dataset ini.

Dari confusion matrix bisa dilihat bahwa spesies Iris-setosa selalu diprediksi dengan benar (100%) oleh kedua model. Hal ini sejalan dengan hasil eksplorasi di post test 1 dimana setosa memiliki ukuran petal yang sangat berbeda dan terpisah jauh dari 2 spesies lainnya.

Jika ada kesalahan prediksi, biasanya terjadi antara Iris-versicolor dan Iris-virginica. Ini karena kedua spesies tersebut memiliki rentang ukuran petal yang saling tumpang tindih (overlap), sehingga model agak kesulitan membedakan di titik-titik perbatasan.

Preprocessing yang sudah dilakukan di post test 2 (penghapusan duplikat, capping outlier, dan penambahan fitur petal_area serta petal_size_encoded) turut membantu karena data menjadi lebih bersih dan informatif.

### Kesimpulan
1. Dataset IRIS yang sudah melewati tahap preprocessing di post test 2 berhasil digunakan sebagai input untuk klasifikasi supervised learning.
2. Dua model yang dipakai (KNN dan SVM) sama-sama memberikan hasil akurasi tinggi. Dari confusion matrix terlihat bahwa sebagian besar data testing berhasil diprediksi dengan benar.
3. Iris-setosa paling mudah dikenali karena ciri fisiknya yang unik, sementara Iris-versicolor dan Iris-virginica lebih sulit karena area distribusinya yang overlap.
4. Penggunaan metrik evaluasi accuracy, precision, recall, f1-score, dan confusion matrix cukup lengkap untuk menilai performa model klasifikasi.